# 🔍 Financial Crime Intelligence — Day 3

**Fraud Detection Application** — Multi-dataset merging, AI Investigations, Advanced Reports, and Interactive AI Chatbot.

This notebook implements a complete fraud detection pipeline:
1. Upload Transactions, Customers, and Watchlist datasets.
2. Apply rule-based fraud scoring.
3. Generate AI investigation reports for HIGH risk transactions.
4. **[NEW]** Generate Executive Summaries, Top Risk Accounts, and Country Risk Analysis.
5. **[NEW]** Chat with an AI assistant loaded with your dataset context.

---

## Section 1: Package Installation

In [ ]:
# ============================================================
# Section 1: Package Installation
# ============================================================

!pip install --quiet pandas numpy matplotlib gradio transformers torch accelerate

print("✅ All packages installed successfully.")

## Section 2: Imports

In [ ]:
# ============================================================
# Section 2: Imports
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
import tempfile
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("\n✅ All imports loaded.")

## Section 3: Fraud Detection Logic

In [ ]:
# ============================================================
# Section 3: Fraud Detection Logic
# ============================================================

HIGH_RISK_COUNTRIES = ["Russia", "Nigeria", "North Korea"]

def compute_risk_score(row: pd.Series) -> int:
    score = 0
    if row.get("amount", 0) > 500_000: score += 50
    if row.get("amount", 0) > 1_000_000: score += 20
    if str(row.get("country", "")) in HIGH_RISK_COUNTRIES: score += 30
    
    # Add a minor watchlist bump if they are flagged in the merged data
    if "watchlist_flag" in row and row["watchlist_flag"] == True:
        score += 50
        
    return min(score, 100)

def classify_risk_level(score: int) -> str:
    if score >= 70: return "HIGH"
    elif score >= 30: return "MEDIUM"
    else: return "LOW"

print("✅ Fraud detection logic defined.")

## Section 4: Model Loading

In [ ]:
# ============================================================
# Section 4: Model Loading
# ============================================================

MODEL_ID = "Qwen/Qwen3-14B"

print(f"⏳ Loading tokenizer and model: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("✅ Model loaded successfully.")

## Section 5: AI Integration (Reports & Chat)

In [ ]:
# ============================================================
# Section 5: AI Integration
# Contains functions for basic reports, advanced reports, and chat.
# ============================================================

# --- 5A. Basic LLM Call Wrapper ---

def ask_llm(system_prompt, user_prompt, max_tokens=512):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template is not None:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"{system_prompt}\n\n{user_prompt}\n\nAnswer:\n"
        
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# --- 5B. Per-Transaction Investigation Reports ---

def generate_investigation_report(amount, country, risk_score, risk_level):
    system_prompt = """You are an expert Financial Crime Investigator.
Strictly format your response using exactly these headings:
- Risk Summary:
- Reason For Alert:
- Potential Risks:
- Recommended Investigation Steps:
- Final Recommendation:"""
    user_prompt = f"Transaction Details:\n- Amount: {amount}\n- Country: {country}\n- Risk Score: {risk_score}\n- Risk Level: {risk_level}\n\nPlease generate the investigation report."
    return ask_llm(system_prompt, user_prompt)

def batch_generate_reports(df: pd.DataFrame):
    if df is None or df.empty: return "No data available to analyze."
    high_risk_df = df[df["risk_level"] == "HIGH"]
    if high_risk_df.empty: return "✅ **No HIGH risk transactions found.**"
    
    reports_md = ["## 🤖 AI Investigation Reports\n"]
    for _, row in high_risk_df.iterrows():
        tx_id = row.get("transaction_id", "Unknown")
        reports_md.append(f"### Transaction: {tx_id}")
        try:
            report = generate_investigation_report(row.get("amount",0), row.get("country",""), row.get("risk_score",0), row.get("risk_level",""))
            reports_md.append(report)
        except Exception as e:
            reports_md.append(f"*Error generating report: {str(e)}*")
        reports_md.append("---")
    return "\n\n".join(reports_md)

# --- 5C. Advanced Reports ---

def generate_advanced_report(df: pd.DataFrame, report_type: str):
    if df is None or df.empty:
        return "Please analyze data first."
        
    # Summarize dataframe to fit context window
    df_summary = df.describe(include='all').to_string()
    high_risk_sample = df[df['risk_level'] == 'HIGH'].head(10).to_string()
    context = f"DATASET SUMMARY:\n{df_summary}\n\nSAMPLE HIGH RISK TRANSACTIONS:\n{high_risk_sample}"
    
    system_prompt = "You are a Chief Compliance Officer analyzing financial crime data."
    
    if report_type == "Executive Summary":
        user_prompt = f"Based on the following data context, generate a concise Executive Summary of fraud activity, overall risk exposure, and major patterns.\n\n{context}"
    elif report_type == "Top Risk Accounts":
        user_prompt = f"Based on the following data context, identify and analyze the top risk accounts. Summarize why they pose a threat.\n\n{context}"
    elif report_type == "Country Risk":
        user_prompt = f"Based on the following data context, generate a Country Risk Analysis summarizing the geographical concentration of fraud.\n\n{context}"
    else:
        return "Unknown report type."
        
    return ask_llm(system_prompt, user_prompt, max_tokens=800)

# --- 5D. Chatbot Interface ---

def chat_with_data(user_message, history, df):
    if df is None or df.empty:
        return "Please upload and analyze datasets first!"
        
    # Convert a lightweight representation of the data for context
    context_df = df.head(50).to_string() # Keep it small to avoid OOM
    system_prompt = f"""You are an interactive AI Financial Crime Analyst.
You have access to the following dataframe of transactions and customer data:

{context_df}

Answer the user's questions about this data concisely and professionally."""

    # Build history context into the prompt if needed, or just ask the LLM directly.
    # For simplicity and context limit, we inject the dataframe into system prompt.
    response = ask_llm(system_prompt, user_message, max_tokens=300)
    return response

print("✅ AI Integration logic defined.")

## Section 6: Analysis Function (Multi-Dataset)

In [ ]:
# ============================================================
# Section 6: Analysis Function
# ============================================================

def generate_risk_chart(df: pd.DataFrame) -> str:
    if df.empty or "risk_level" not in df.columns: return None
    risk_order = ["LOW", "MEDIUM", "HIGH"]
    counts = df["risk_level"].value_counts().reindex(risk_order, fill_value=0)
    colors = ["#2ecc71", "#f39c12", "#e74c3c"]
    fig, ax = plt.subplots(figsize=(8, 5), facecolor="#1a1a2e")
    ax.set_facecolor("#1a1a2e")
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=0.8, width=0.6)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2.0, height + 0.3, f"{int(height)}", ha="center", va="bottom", fontweight="bold", fontsize=14, color="white")
    ax.set_title("Risk Level Distribution", fontsize=18, fontweight="bold", color="white", pad=15)
    ax.set_xlabel("Risk Level", fontsize=13, color="white", labelpad=10)
    ax.set_ylabel("Number of Transactions", fontsize=13, color="white", labelpad=10)
    ax.tick_params(axis="both", colors="white", labelsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#444")
    ax.spines["bottom"].set_color("#444")
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    chart_path = os.path.join(tempfile.gettempdir(), "risk_distribution.png")
    fig.savefig(chart_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    return chart_path

def analyze_all_datasets(tx_file, cust_file, watch_file):
    if tx_file is None:
        return ("⚠️ Transactions file is required.", pd.DataFrame(), None)
        
    try:
        df = pd.read_csv(tx_file)
        
        # Merge Customers if available
        if cust_file is not None:
            df_cust = pd.read_csv(cust_file)
            if 'account' in df.columns and 'account' in df_cust.columns:
                df = df.merge(df_cust, on="account", how="left")
                
        # Merge Watchlist if available
        if watch_file is not None:
            df_watch = pd.read_csv(watch_file)
            df_watch["watchlist_flag"] = True
            if 'account' in df.columns and 'account' in df_watch.columns:
                df = df.merge(df_watch[["account", "watchlist_flag"]], on="account", how="left")
            elif 'customer_name' in df.columns and 'customer_name' in df_watch.columns:
                df = df.merge(df_watch[["customer_name", "watchlist_flag"]], on="customer_name", how="left")
            df["watchlist_flag"] = df["watchlist_flag"].fillna(False)
            
    except Exception as e:
        return (f"❌ Error reading/merging CSVs: {e}", pd.DataFrame(), None)

    df["amount"] = pd.to_numeric(df.get("amount", 0), errors="coerce").fillna(0)
    if "country" in df.columns:
        df["country"] = df["country"].astype(str).str.strip()
        
    df["risk_score"] = df.apply(compute_risk_score, axis=1)
    df["risk_level"] = df["risk_score"].apply(classify_risk_level)
    
    total = len(df)
    high_count = int((df["risk_level"] == "HIGH").sum())
    medium_count = int((df["risk_level"] == "MEDIUM").sum())
    low_count = int((df["risk_level"] == "LOW").sum())
    summary = (f"📊  SUMMARY METRICS\n{'═' * 40}\n  Total Transactions : {total}\n  🔴 High Risk       : {high_count}\n  🟡 Medium Risk     : {medium_count}\n  🟢 Low Risk        : {low_count}\n{'═' * 40}")
    
    chart_path = generate_risk_chart(df)
    return summary, df, chart_path

print("✅ Multi-dataset analysis function defined.")

## Section 7: UI Integration (Gradio Tabs)

In [ ]:
# ============================================================
# Section 7: UI Integration
# ============================================================

theme = gr.themes.Base(primary_hue="blue", neutral_hue="slate", font=gr.themes.GoogleFont("Inter"))
demo = gr.Blocks(theme=theme, title="Financial Crime Intelligence")

# We will use this hidden state to hold the merged DataFrame for the LLM reports & chat
global_df_state = gr.State()

with demo:
    gr.Markdown("""# 🛡️ Financial Crime Intelligence\n### Multi-Dataset Fraud Engine & AI Assistant""")
    
    with gr.Tabs():
        # --- TAB 1: Data & Analysis ---
        with gr.Tab("1. Data & Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    tx_upload = gr.File(label="📁 Transactions CSV (Required)", file_types=[".csv"], type="filepath")
                    cust_upload = gr.File(label="📁 Customers CSV (Optional)", file_types=[".csv"], type="filepath")
                    watch_upload = gr.File(label="📁 Watchlist CSV (Optional)", file_types=[".csv"], type="filepath")
                    analyze_btn = gr.Button("🔍 Merge & Analyze", variant="primary", size="lg")
                with gr.Column(scale=2):
                    summary_output = gr.Textbox(label="📊 Summary Statistics", lines=8, interactive=False)
                    chart_output = gr.Image(label="Risk Level Distribution", type="filepath")
            
            gr.Markdown("### 📋 Scored Results")
            results_table = gr.Dataframe(label="Merged Results", interactive=False)

        # --- TAB 2: AI Reports ---
        with gr.Tab("2. AI Reports"):
            with gr.Row():
                btn_exec = gr.Button("📈 Executive Summary")
                btn_top_risk = gr.Button("🚨 Top Risk Accounts")
                btn_country = gr.Button("🌍 Country Risk Analysis")
            
            gr.Markdown("### Per-Transaction Investigation")
            btn_tx_reports = gr.Button("🤖 Generate AI Investigation Reports for High Risk", variant="secondary")
            
            reports_output = gr.Markdown(label="Generated Report")
            
        # --- TAB 3: AI Chatbot ---
        with gr.Tab("3. AI Assistant Chat"):
            gr.Markdown("Ask questions like: *Why was account X flagged?* or *Summarize fraud activity.*");
            
            chatbot = gr.Chatbot(height=400)
            msg = gr.Textbox(label="Ask the AI about your data:")
            clear = gr.ClearButton([msg, chatbot])
            
            def user_msg(user_message, history):
                return "", history + [[user_message, None]]
                
            def bot_msg(history, df):
                user_message = history[-1][0]
                bot_response = chat_with_data(user_message, history, df)
                history[-1][1] = bot_response
                return history
                
            msg.submit(user_msg, [msg, chatbot], [msg, chatbot], queue=False).then(
                bot_msg, [chatbot, global_df_state], chatbot
            )

    # Event Wiring
    analyze_btn.click(
        fn=analyze_all_datasets, 
        inputs=[tx_upload, cust_upload, watch_upload], 
        outputs=[summary_output, global_df_state, chart_output]
    ).then(
        fn=lambda df: df, 
        inputs=[global_df_state], 
        outputs=[results_table]
    )
    
    btn_exec.click(fn=lambda df: generate_advanced_report(df, "Executive Summary"), inputs=[global_df_state], outputs=[reports_output])
    btn_top_risk.click(fn=lambda df: generate_advanced_report(df, "Top Risk Accounts"), inputs=[global_df_state], outputs=[reports_output])
    btn_country.click(fn=lambda df: generate_advanced_report(df, "Country Risk"), inputs=[global_df_state], outputs=[reports_output])
    btn_tx_reports.click(fn=batch_generate_reports, inputs=[global_df_state], outputs=[reports_output])

print("✅ Gradio UI built.")

## Section 8: Launch Application

In [ ]:
# ============================================================
# Section 8: Launch Application
# ============================================================

demo.launch(inline=True, share=False)